# Notebook 06 — k-Nearest Neighbors and Naive Bayes

**Module:** 13 — Machine Learning for Biology  
**Tier:** 1 — Master  
**Notebook:** 06 of 15  
**Time estimate:** 75 minutes

> kNN has no training step — it's pure memorization. Naive Bayes makes one
> strong assumption (feature independence) and works surprisingly well.
> Both are instructive precisely because of their simplicity.

---
## Step 1 — Motivation

kNN is the most intuitive algorithm: classify a new point by looking at its nearest
neighbors in the training set. In single-cell biology, this is exactly how cell type
annotation works — "this new cell looks like its k nearest cells in the reference atlas."
Naive Bayes appears in spam filtering, text classification, and RNA-seq cell typing.

---
## Step 2 — Intuition

**kNN:** Store all training data. For a new point $\mathbf{x}$:
1. Compute distance to all training points
2. Find the $k$ nearest neighbors
3. Return the majority class (classification) or mean (regression)

**k trade-off:**
- $k=1$: memorizes training data, zero training error, high variance
- Large $k$: smooth decision boundary, higher bias
- Optimal $k$ ≈ $\sqrt{n}$ as a rough rule

**Curse of dimensionality:** In high dimensions, distances become meaningless.
If $p=1000$ and all features are i.i.d. uniform, the ratio of nearest to farthest
neighbor distance → 1 as $p \to \infty$. kNN requires either:
- Dimensionality reduction first (PCA → kNN, as in scRNA-seq)
- Feature selection

**Naive Bayes:** Applies Bayes' theorem with a strong independence assumption:
$$P(y | \mathbf{x}) \propto P(y) \prod_j P(x_j | y)$$

"Naive" = assumes all features are conditionally independent given the class.
This is almost never true biologically — yet Naive Bayes often performs well because
it needs to estimate far fewer parameters.

---
## Step 3 — Biological Background

**kNN in single-cell biology:**
- Seurat, Scanpy, and UMAP all build k-nearest-neighbor graphs on the PCA representation
- Cell type annotation: reference atlas trained, new cells labeled by kNN vote among
  reference cells
- Community detection (Leiden) operates on the kNN graph → NB06 (Module 12)

**Distance metrics for biological data:**
- **Euclidean:** Works on normalized expression (PCA space)
- **Cosine similarity:** $\cos(\theta) = \mathbf{x}^T\mathbf{y} / (\|\mathbf{x}\|\|\mathbf{y}\|)$.
  For bag-of-words text features or k-mer frequency vectors — direction matters, not magnitude
- **Hamming distance:** For binary/categorical features (genetic variants: 0/1/2 allele copies)
- **Manhattan (L1):** More robust to outliers than Euclidean

**Naive Bayes for cell typing:**
- Features: expression level of marker genes (continuous → Gaussian NB)
- Or discretized expression: high/low per gene (Bernoulli NB)
- Fast to train, interpretable, works with missing features

**Complement NB for RNA-seq:** Models the complement class distribution, works better
with imbalanced classes — used in text classification and applied to metagenomics
(classify reads by species from k-mer frequency)

---
## Step 4 — Mathematical Explanation

**kNN classification:**
$$\hat{y} = \arg\max_c \sum_{i \in \mathcal{N}_k(\mathbf{x})} \mathbf{1}[y_i = c]$$

**Distance weighted kNN:** Weight vote by $1/d_i$ — closer neighbors have more influence.

**Curse of dimensionality — formal:**
Let $R_{(1)} \leq \ldots \leq R_{(n)}$ be sorted distances in $\mathbb{R}^p$.
For uniform data on $[0,1]^p$, $R_{(1)} / R_{(n)} \to 1$ as $p \to \infty$.
All points become equidistant — kNN's distances are meaningless.

**Naive Bayes — Gaussian:**
$$P(x_j | y=c) = \mathcal{N}(x_j; \mu_{jc}, \sigma_{jc}^2)$$

Parameters estimated by MLE: $\mu_{jc} = \bar{x}_{jc}$, $\sigma_{jc}^2 = \text{var}(x_j | y=c)$.

**Log posterior:**
$$\log P(y=c | \mathbf{x}) \propto \log P(y=c) + \sum_j \log P(x_j | y=c)$$

Classification: $\hat{y} = \arg\max_c \log P(y=c | \mathbf{x})$.

**Laplace smoothing (for discrete features):**
$$P(x_j=v | y=c) = \frac{N_{jvc} + \alpha}{N_c + \alpha |V_j|}$$
prevents zero probabilities when a feature value is unseen in training.

In [ ]:
# Step 6 — Python: kNN and Naive Bayes from scratch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score

# ---- kNN from scratch ----
class KNNScratch:
    def __init__(self, k=5, metric='euclidean'):
        self.k = k
        self.metric = metric
    
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self
    
    def _distance(self, a, b):
        if self.metric == 'euclidean':
            return np.sqrt(np.sum((a - b)**2, axis=-1))
        elif self.metric == 'manhattan':
            return np.sum(np.abs(a - b), axis=-1)
        elif self.metric == 'cosine':
            dot = a @ b.T
            norm = np.linalg.norm(a, axis=-1, keepdims=True) * np.linalg.norm(b, axis=-1)
            return 1 - dot / (norm + 1e-10)
    
    def predict(self, X):
        predictions = []
        for x in X:
            dists = self._distance(self.X_train, x[np.newaxis, :]).ravel()
            nn_idx = np.argsort(dists)[:self.k]
            nn_labels = self.y_train[nn_idx]
            labels, counts = np.unique(nn_labels, return_counts=True)
            predictions.append(labels[np.argmax(counts)])
        return np.array(predictions)

# ---- Gaussian Naive Bayes from scratch ----
class GaussianNBScratch:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.priors = {c: np.mean(y == c) for c in self.classes}
        self.mu = {c: X[y==c].mean(axis=0) for c in self.classes}
        self.sigma2 = {c: X[y==c].var(axis=0) + 1e-9 for c in self.classes}
        return self
    
    def log_posterior(self, X):
        log_probs = []
        for c in self.classes:
            log_prior = np.log(self.priors[c])
            # log N(x; mu, sigma^2) = -0.5*log(2*pi*sigma^2) - (x-mu)^2/(2*sigma^2)
            log_likelihood = -0.5 * np.sum(
                np.log(2 * np.pi * self.sigma2[c]) + (X - self.mu[c])**2 / self.sigma2[c],
                axis=1)
            log_probs.append(log_prior + log_likelihood)
        return np.column_stack(log_probs)
    
    def predict(self, X):
        log_probs = self.log_posterior(X)
        return self.classes[np.argmax(log_probs, axis=1)]
    
    def predict_proba(self, X):
        log_probs = self.log_posterior(X)
        log_probs -= log_probs.max(axis=1, keepdims=True)
        probs = np.exp(log_probs)
        return probs / probs.sum(axis=1, keepdims=True)

# ---- Generate single-cell gene expression dataset (5 cell types) ----
rng = np.random.default_rng(42)

def make_cell_type_data(n_per_type=80, n_genes=50, seed=42):
    rng = np.random.default_rng(seed)
    # 5 cell types, each with distinct marker genes
    means = np.zeros((5, n_genes))
    for i in range(5):
        # Each cell type has 5 highly expressed marker genes
        marker_genes = np.arange(i*10, i*10+10) % n_genes
        means[i, marker_genes] = 3.0
    
    X = []
    y = []
    for ct in range(5):
        X.append(rng.normal(means[ct], 0.8, (n_per_type, n_genes)).clip(0))
        y.extend([ct] * n_per_type)
    return np.vstack(X), np.array(y)

X_cell, y_cell = make_cell_type_data()
X_tr, X_te, y_tr, y_te = train_test_split(X_cell, y_cell, test_size=0.2, random_state=42, stratify=y_cell)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)

# Compare kNN (k=5, 15) and GNB
for name, clf in [
    ('kNN (scratch, k=5)', KNNScratch(k=5)),
    ('kNN (scratch, k=15)', KNNScratch(k=15)),
    ('GNB (scratch)', GaussianNBScratch()),
    ('kNN (sklearn, k=5)', KNeighborsClassifier(k=5 if False else 5)),  # use n_neighbors
    ('GNB (sklearn)', GaussianNB()),
]:
    if 'kNN (sklearn' in name:
        clf = KNeighborsClassifier(n_neighbors=5)
    clf.fit(X_tr_sc, y_tr)
    acc = (clf.predict(X_te_sc) == y_te).mean()
    print(f'{name:<35}: test accuracy = {acc:.4f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Dimensionality reduction for visualization
from sklearn.decomposition import PCA
pca2 = PCA(n_components=2)
X_2d = pca2.fit_transform(X_tr_sc)
X_te_2d = pca2.transform(X_te_sc)

# Panel A: kNN decision boundary (2D PCA, k=5)
ax = axes[0]
knn_2d = KNeighborsClassifier(n_neighbors=5)
knn_2d.fit(X_2d, y_tr)
xx, yy = np.meshgrid(np.linspace(X_2d[:,0].min()-1, X_2d[:,0].max()+1, 200),
                       np.linspace(X_2d[:,1].min()-1, X_2d[:,1].max()+1, 200))
Z = knn_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z, alpha=0.3, cmap='tab10', levels=np.arange(-0.5, 5.5, 1))
cmap = plt.cm.get_cmap('tab10', 5)
for ct in range(5):
    mask = y_tr == ct
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=[cmap(ct)], s=15, alpha=0.6, label=f'Type {ct}')
ax.set_title('A. kNN (k=5) on PCA-reduced\ncell type data')
ax.legend(fontsize=7, loc='upper right')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

# Panel B: k vs. accuracy
ax = axes[1]
k_values = range(1, 51, 2)
cv_scores = [cross_val_score(KNeighborsClassifier(n_neighbors=k),
                                X_tr_sc, y_tr, cv=5).mean() for k in k_values]
ax.plot(k_values, cv_scores, 'steelblue', lw=2, marker='o', ms=3)
best_k = k_values[np.argmax(cv_scores)]
ax.axvline(best_k, color='red', ls='--', lw=1, label=f'Best k={best_k}')
ax.set_xlabel('k (number of neighbors)')
ax.set_ylabel('5-fold CV accuracy')
ax.set_title('B. Accuracy vs. k\n(bias-variance tradeoff in kNN)')
ax.legend(fontsize=8)

# Panel C: Naive Bayes probability calibration
ax = axes[2]
gnb = GaussianNBScratch()
gnb.fit(X_tr_sc, y_tr)
probs = gnb.predict_proba(X_te_sc)
max_probs = probs.max(axis=1)
correct = (gnb.predict(X_te_sc) == y_te)
bins = np.linspace(0, 1, 11)
bin_acc = [correct[(max_probs >= bins[i]) & (max_probs < bins[i+1])].mean()
             if np.any((max_probs >= bins[i]) & (max_probs < bins[i+1])) else np.nan
             for i in range(len(bins)-1)]
bin_centers = (bins[:-1] + bins[1:]) / 2
ax.bar(bin_centers, bin_acc, width=0.08, color='steelblue', alpha=0.7)
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
ax.set_xlabel('Predicted confidence (max prob)')
ax.set_ylabel('Actual accuracy')
ax.set_title('C. Naive Bayes calibration\n(confidence vs. accuracy)')
ax.legend(fontsize=8)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)

plt.suptitle('Module 13 NB06: k-Nearest Neighbors and Naive Bayes', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('knn_naive_bayes.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Add distance-weighted voting to `KNNScratch` (vote weight = 1/distance).
   Does it improve performance on the cell-type dataset?
2. Demonstrate the curse of dimensionality: add random noise features to the
   dataset, increasing $p$ from 50 to 500. How does kNN (k=5) accuracy change?
3. Implement Bernoulli Naive Bayes for binary (discretized) gene expression data.
   Test on the cell-type dataset with threshold `X > mean` → 1, else 0.
4. Naive Bayes assumes feature independence. For gene expression data, is this
   reasonable? What happens to NB when two features are highly correlated?

---
## Step 10 — Quiz

1. What are the two steps of kNN prediction (no training step!)?
2. What is the curse of dimensionality? Why does it hurt kNN?
3. Write the Naive Bayes classifier formula. What does 'naive' mean?
4. What is Laplace smoothing and when is it needed?
5. What distance metric is appropriate for gene expression data after PCA?

---
## Step 12 — Reflection

> *[In one paragraph: explain why Naive Bayes often works well in practice despite
> its independence assumption being violated, using the concept of optimal decision
> boundaries and what matters for classification accuracy.]*

---
*Next: `07_ensemble_methods_gradient_boosting.ipynb`*